# TP4 - Assembler et entrainer un mini-Transformer

Ce notebook assemble embeddings, attention multi-tetes, blocs pre-norm, loss autoregressive et generation a contexte limite. `QUICK_RUN` permet une validation rapide sur CPU.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)
path = Path('input.txt')
if not path.exists():
    urlretrieve('https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt', path)
text = path.read_text(encoding='utf-8')
chars = sorted(set(text)); vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}; itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]; decode = lambda ids: ''.join(itos[i] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]

QUICK_RUN = True
batch_size, block_size = 16, 32
max_iters = 200 if QUICK_RUN else 5000
eval_interval, eval_iters = 50, 20 if QUICK_RUN else 200
learning_rate, n_embd, n_head, n_layer, dropout = 1e-3, 64, 4, 4, 0.0
device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert n_embd % n_head == 0

def get_batch(split):
    source = train_data if split == 'train' else val_data
    starts = torch.randint(len(source) - block_size, (batch_size,))
    return (torch.stack([source[i:i + block_size] for i in starts]).to(device),
            torch.stack([source[i + 1:i + block_size + 1] for i in starts]).to(device))
print(device, 'vocab=', vocab_size, 'head_size=', n_embd // n_head)

cpu vocab= 65 head_size= 16


In [2]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)
        wei = q @ k.transpose(-2, -1) * k.size(-1) ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = self.dropout(F.softmax(wei, dim=-1))
        return wei @ self.value(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))

    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.sa = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedForward(n_embd)
        self.ln1, self.ln2 = nn.LayerNorm(n_embd), nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [3]:
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = self.blocks(tok_emb + pos_emb)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(B * T, vocab_size), targets.reshape(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat((idx, torch.multinomial(probs, 1)), dim=1)
        return idx

@torch.no_grad()
def estimate_loss(model):
    result = {}
    model.eval()
    for split in ('train', 'val'):
        values = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split); _, loss = model(xb, yb); values[k] = loss.item()
        result[split] = values.mean().item()
    model.train()
    return result

In [4]:
model = BigramLanguageModel().to(device)
print('parameters:', sum(p.numel() for p in model.parameters()))
print('before training:')
print(decode(model.generate(torch.zeros((1, 1), dtype=torch.long, device=device), 200)[0].tolist()))
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
history = []
for iteration in range(max_iters):
    if iteration % eval_interval == 0:
        metrics = estimate_loss(model); history.append((iteration, metrics)); print(iteration, metrics)
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
print('after training:')
print(decode(model.generate(torch.zeros((1, 1), dtype=torch.long, device=device), 500)[0].tolist()))

parameters: 209729
before training:



DuJAMJ3fX$VWoGCYngj$'onoE,$AT-NvynticCzJ-c!bnDNyjwND,?to;
VcANzq:nU$3NVAz:YzgNkceei.-efVhpXz,NApQe.3hRfNThglUzoecXgZ q.g?d
xeo$ug:'QmZqRzF!neLZmh.TW,$NJko.iu!seNwol$zJipJhwiKeo,Ssnu3blKpSpbMcSWhWmYn&U


0 {'train': 4.4128007888793945, 'val': 4.412615776062012}


50 {'train': 2.9506497383117676, 'val': 2.9982378482818604}


100 {'train': 2.647423267364502, 'val': 2.664402484893799}


150 {'train': 2.5855891704559326, 'val': 2.570140838623047}


after training:



Ant the irer nougo Qsou f d eow,
JThy hhin.

OCKIUCKIst
Nal PURSAN, tnhar:
Me dethaseoid m:
I, theareno Yhat ttomafangbe y.

Bu carulbreman thersomp Abrd noun thy la wauree nd ithazillifou Cingbe wen.
ARGE! aresa lear poutom e sthatee mive-t med d hanuthere'ly
Fhunpak? indis s

INinime!TYpe bnd maiworthar d.

Wheceangead m.
ap apoDath
Bous bersill beay lene arinORENfautre,

ARU.
KKENDETEUKE3D&e tomanoumour he h tho oethto erdeehadoncoat ay aesthrer t this's.


Mamiee hig BUENEDINRO:
STMEAnx fito


### Questions et reponses

1. **Preparation:** `dh = n_embd / n_head = 64 / 4 = 16`; un batch contient `16 * 32 = 512` tokens. Sur CPU, reduire d'abord `max_iters`, puis `eval_iters`, `batch_size` ou `n_layer` pour un test rapide.
2. **Tete d'attention:** pour `(B,T,C)=(16,32,64)`, `q,k,v` ont la forme `(16,32,16)`, `wei` `(16,32,32)` et la sortie `(16,32,16)`. `k.size(-1)` est `dh`, la largeur reelle de la tete, et `tril[:T,:T]` permet de traiter un contexte plus court que `block_size`.
3. **Multi-tetes:** on concatene sur la derniere dimension pour obtenir `4 * 16 = 64`, puis la projection `W_O` remelange les informations. Si `n_embd` n'est pas divisible par `n_head`, chaque tete ne peut pas avoir une largeur entiere commune.
4. **Bloc:** l'ordre pre-norm est `x = x + attention(LayerNorm(x))`, puis `x = x + MLP(LayerNorm(x))`. L'attention est la communication entre positions; le MLP est le calcul individuel de chaque token.
5. **Embeddings et logits:** `token_embedding_table` a la forme `(vocab_size,n_embd)`, `position_embedding_table` `(block_size,n_embd)` et `lm_head` `(n_embd,vocab_size)`. Le token embedding encode l'identite, la position encode l'ordre et `lm_head` reconvertit l'etat cache en scores du vocabulaire; il n'est donc plus une table `C x C`.
6. **Broadcasting:** `tok_emb` est `(B,T,n_embd)` et `pos_emb` `(T,n_embd)`. Broadcasting aligne les deux dernieres dimensions et ajoute le meme vecteur de position a chaque element du batch. Les logits restent `(B,T,vocab_size)` pendant la generation afin de conserver l'axe temporel; on selectionne seulement `logits[:, -1, :]`.
7. **Generation:** `@torch.no_grad()` economise memoire et calcul car aucun gradient n'est necessaire. `idx[:, -block_size:]` respecte la limite de contexte. `multinomial` echantillonne selon la distribution et produit de la diversite, alors que `argmax` est deterministe et repetitif.
8. **Evaluation et apprentissage:** `model.eval()` desactive notamment dropout pendant la mesure; `model.train()` le reactive ensuite. Si la loss train baisse mais la validation remonte, le modele surapprend. `cross_entropy` compare les logits aux cibles aplaties `(B*T)` et le gradient met a jour toutes les briques du modele.
9. **Experiences:** avec `QUICK_RUN=True`, le notebook verifie rapidement le pipeline. Pour l'experience complete du sujet, passer a `False`, puis comparer `block_size=16/64`, `n_head=1/4` a `n_embd` fixe et plusieurs valeurs de `dropout`. Une generation imparfaite est normale avec peu d'iterations.
10. **Bilan global:** l'attention cree le contexte, LayerNorm stabilise chaque token, le MLP transforme ce token, les connexions residuelles preservent l'information et `lm_head` produit les logits. La loss agit sur toutes les positions et retropropage son signal a l'ensemble du Transformer.